In [61]:
import warnings 
import pandas as pd
import seaborn as sns
import networkx as nx
import matplotlib.pyplot as plt

from glob import glob 

warnings.filterwarnings('ignore')
%matplotlib inline
%load_ext autoreload
%autoreload 2

pd.options.display.float_format = "{:,.3f}".format
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

INDIR_DPCLUST = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/dpclust_ccf'
INDIR_CONIPHER = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/angel/conipher_trees'
INFILE_MUTATIONS = '/home/grace/work/PPCG_DifferentialGenesetMutation/outputs/angel_mutations_all_donors/variant_processing/mutations.filtered.tsv'
INFILE_SHEET = '/home/grace/work/PPCG_DifferentialGenesetMutation/samplesheet.angel.alldonors.tsv'
OUTFILE_MASTER = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/manual/mutations.assigned.160326.tsv'
# PPCG0213a has no mutations. 
# Some PPCG donors are multisample primary. 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


***Donor Summary Information & Handling Method***

load & augmenting the workflow samplesheet

In [23]:
def format_files(missing_text: str) -> set[str]:
    expected = set(['SNV', 'INDEL', 'CNA', 'SV'])
    if missing_text == '.':
        return expected
    missing = set(missing_text.split(','))
    return expected-missing

sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
sheet = sheet.sort_values('sample')

# annotating which files are present
vclasses = ['SNV', 'INDEL', 'CNA', 'SV']
sheet['Files'] = sheet['Missing_Files'].apply(format_files)
for vclass in vclasses:
    sheet[vclass] = sheet['Files'].apply(lambda x: vclass in x)
sheet = sheet.drop(['Files'], axis=1)
print(sheet.tail())

         sample     donor      tissue  purity  ploidy    WGD cohort Missing_Files   SNV  INDEL   CNA    SV
1149  PPCG1102a  PPCG1102     Primary   0.384   4.720   True   PPCG             .  True   True  True  True
1150  PPCG1103a  PPCG1103     Primary   0.346   2.636  False   PPCG             .  True   True  True  True
1151  PPCG1105a  PPCG1105     Primary   0.252   1.839  False   PPCG             .  True   True  True  True
1152  PPCG1106a  PPCG1106     Primary   0.709   1.690  False   PPCG             .  True   True  True  True
219   PPCG1107a  PPCG1107  Metastasis   0.885   3.533   True  COMBI             .  True   True  True  True


Generate summary metadata for each donor. 

In [24]:
DPC_DONORS = set([x.split('/')[-1][:8] for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")])
TREE_DONORS = set([x.split('/')[-2][:8] for x in glob(f"{INDIR_CONIPHER}/*/allTrees.txt")])

data = []
cohort_LUT = sheet.drop_duplicates('donor').set_index('donor')['cohort'].to_dict()
tissue_LUT = sheet.set_index('sample')['tissue'].to_dict()
primaries_LUT = sheet[sheet['tissue']=='Primary'].groupby('donor')['sample'].agg(list).to_dict()
secondaries_LUT = sheet[sheet['tissue']=='Metastasis'].groupby('donor')['sample'].agg(list).to_dict()

for vclass in vclasses:
    for donor in sheet['donor'].unique():
        label = f"{donor}|{vclass}"
        cohort = cohort_LUT[donor]
        temp = sheet[(sheet['donor']==donor) & (sheet[vclass]==True)]
        samples = temp['sample'].nunique()
        primary = temp[temp['tissue']=='Primary']['sample'].nunique()
        secondary = temp[temp['tissue']=='Metastasis']['sample'].nunique()
        sample_list = sorted(temp['sample'].unique())
        primary_list = sorted(temp[temp['tissue']=='Primary']['sample'].unique())
        secondary_list = sorted(temp[temp['tissue']=='Metastasis']['sample'].unique())
        data.append((label, donor, vclass, cohort, samples, primary, secondary, sample_list, primary_list, secondary_list))
hframe = pd.DataFrame(data, columns=['label', 'donor', 'vclass', 'cohort', 'samples', 'primary', 'secondary', 'sample_list', 'primary_list', 'secondary_list'])
hframe['dpclust'] = hframe['donor'].apply(lambda x: x in DPC_DONORS)
hframe['conipher'] = hframe['donor'].apply(lambda x: x in TREE_DONORS)

print(hframe.drop_duplicates('donor').groupby('dpclust')['conipher'].value_counts(dropna=False))
hframe.head()

dpclust  conipher
False    False       903
True     True         75
         False         4
Name: count, dtype: int64


,label,donor,vclass,cohort,samples,primary,secondary,sample_list,primary_list,secondary_list,dpclust,conipher
0,PPCG0001|SNV,PPCG0001,SNV,PPCG,1,1,0,[PPCG0001a],[PPCG0001a],[],False,False
1,PPCG0002|SNV,PPCG0002,SNV,PPCG,1,1,0,[PPCG0002a],[PPCG0002a],[],False,False
2,PPCG0003|SNV,PPCG0003,SNV,PPCG,1,1,0,[PPCG0003a],[PPCG0003a],[],False,False
3,PPCG0004|SNV,PPCG0004,SNV,PPCG,1,1,0,[PPCG0004a],[PPCG0004a],[],False,False
4,PPCG0005|SNV,PPCG0005,SNV,PPCG,1,1,0,[PPCG0005a],[PPCG0005a],[],False,False


Used metadata to select which lineage / clone annotation method to use. 

In [25]:
# Decide which method to use when assigning variants (of each type) per donor. 
def get_handling(row: pd.Series) -> str:
    if row['primary'] == 0 and row['secondary'] == 0:
        return 'ignore'
    elif row['primary'] == 1 and row['secondary'] == 0:
        return 'single_primary'
    elif row['primary'] >= 2 and row['secondary'] == 0:
        return 'multi_primary'
    elif row['primary'] == 0 and row['secondary'] == 1:
        return 'single_met'
    elif row['primary'] == 0 and row['secondary'] >= 2:
        return 'multi_met'
    # elif row['dpclust'] == True:
    #     return 'dpclust'
    elif row['primary'] == 1 and row['secondary'] == 1:
        return 'single_primary_single_met'
    elif row['primary'] == 1 and row['secondary'] >= 2:
        return 'single_primary_multi_met'
    raise NotImplementedError(row)

hframe['handling'] = hframe.apply(get_handling, axis=1)

print('PPCG')
print(hframe[hframe['cohort']=='PPCG'].groupby('vclass')['handling'].value_counts(dropna=False).unstack().fillna(0).astype(int))
print()
print('COMBI')
print(hframe[hframe['cohort']=='COMBI'].groupby('vclass')['handling'].value_counts(dropna=False).unstack().fillna(0).astype(int))


PPCG
handling  ignore  multi_primary  single_primary
vclass                                         
CNA            0             21             859
INDEL         22             20             838
SNV            3             21             856
SV            45             17             818

COMBI
handling  ignore  multi_met  single_met  single_primary  single_primary_multi_met  single_primary_single_met
vclass                                                                                                      
CNA            0          7          22               0                        10                         63
INDEL          2          8          21               3                         8                         60
SNV            0          7          22               0                        10                         63
SV             6          7          20               3                         8                         58


***Loading and preprocessing mutations***

In [26]:
def load_mutations() -> pd.DataFrame:
    df = pd.read_csv(INFILE_MUTATIONS, sep='\t', header=0)
    sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
    sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
    valid = set(sheet['sample'].unique())
    df = df[df['sample'].isin(valid)].copy()

    donor2cohort = sheet.drop_duplicates('donor').set_index('donor')['cohort'].to_dict()
    sample2tissue = sheet.drop_duplicates('sample').set_index('sample')['tissue'].to_dict()
    df['cohort'] = df['donor'].map(donor2cohort)
    df['tissue'] = df['sample'].map(sample2tissue)
    return df 

def preprocess_mutations(table: pd.DataFrame) -> pd.DataFrame:
    df = table.copy()

    # drop duplicates. 
    # sorting ensures higher ccf replicates are kept. 
    df = df.sort_values(['est_ccf', 'coords'], ascending=False)
    # SNV/INDEL: exact variant duplicates. 
    shard1 = df[df['vclass'].isin(['SNV', 'INDEL'])].copy()
    shard1 = shard1.drop_duplicates(subset=['sample', 'vclass', 'coords'])
    # SV/CNA: gene level duplicates (ie if PPCG0001a has 2 SVs in the ERG gene)
    shard2 = df[df['vclass'].isin(['SV', 'CNA'])].copy()
    shard2 = shard2.drop_duplicates(subset=['sample', 'vclass', 'gene'])
    # recombine 
    df = pd.concat([shard1, shard2], ignore_index=True)

    # derive identifier 'ident' for each variant
    mask = df['vclass'].isin(['SV', 'CNA'])
    df.loc[mask, 'ident'] = df['donor'] + '|' + df['vclass'] + '|' + df['gene']
    df.loc[~mask, 'ident'] = df['donor'] + '|' + df['vclass'] + '|' + df['coords']
    df['label'] = df['donor'] + '|' + df['vclass']

    # dfate clonality
    # df['clonality'] = df['est_ccf'].apply(lambda x: 'subclonal' if x<0.9 else 'clonal')

    # formatting
    df = df.rename(columns={'est_ccf': 'variant_ccf'})
    # df = df[['ident', 'label', 'gene', 'vclass', 'cohort', 'donor', 'sample', 'tissue', 'variant_ccf']].copy()
    df = df.copy()
    df = df.sort_values('ident')
    return df

muts = load_mutations()
print(muts.head())
muts = preprocess_mutations(muts)
muts.head()

      sample     donor                   coords ref alt vclass vtype                 annotation  est_ccf     gene cohort   tissue
0  PPCG0322a  PPCG0322  3:132062590-21:42839615   .   .     SV   TRA                gene_fusion      NaN     ACP3   PPCG  Primary
1  PPCG0322a  PPCG0322  3:132062590-21:42839615   .   .     SV   TRA                gene_fusion      NaN  TMPRSS2   PPCG  Primary
2  PPCG0322a  PPCG0322   4:153179188-5:71525815   .   .     SV   TRA        transcript_ablation      NaN   MRPS27   PPCG  Primary
3  PPCG0322a  PPCG0322      9:9618517-9:9619627   .   .     SV   INV  bidirectional_gene_fusion      NaN    PTPRD   PPCG  Primary
4  PPCG0322a  PPCG0322      9:9474532-9:9619633   .   .     SV   INV  bidirectional_gene_fusion      NaN    PTPRD   PPCG  Primary


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label
52168,PPCG0001a,PPCG0001,11:5153383,GCC,G,INDEL,INDEL,frameshift_variant,0.880,OR52A5,PPCG,Primary,PPCG0001|INDEL|11:5153383,PPCG0001|INDEL
9449,PPCG0001a,PPCG0001,4:186071244,GCC,G,INDEL,INDEL,3_prime_UTR_variant,1.000,SLC25A4,PPCG,Primary,PPCG0001|INDEL|4:186071244,PPCG0001|INDEL
52235,PPCG0001a,PPCG0001,10:75562095,C,T,SNV,SNV,3_prime_UTR_variant,0.880,NDST2,PPCG,Primary,PPCG0001|SNV|10:75562095,PPCG0001|SNV
30793,PPCG0001a,PPCG0001,11:76174922,C,T,SNV,SNV,missense_variant,1.000,EMSY,PPCG,Primary,PPCG0001|SNV|11:76174922,PPCG0001|SNV
50413,PPCG0001a,PPCG0001,12:109251359,T,A,SNV,SNV,5_prime_UTR_variant,0.890,SSH1,PPCG,Primary,PPCG0001|SNV|12:109251359,PPCG0001|SNV


In [27]:
sheet.head()

,sample,donor,tissue,purity,ploidy,WGD,cohort,Missing_Files,SNV,INDEL,CNA,SV
220,PPCG0001a,PPCG0001,Primary,0.280,1.886,False,PPCG,.,True,True,True,True
221,PPCG0002a,PPCG0002,Primary,0.498,2.311,False,PPCG,.,True,True,True,True
222,PPCG0003a,PPCG0003,Primary,0.778,2.385,False,PPCG,.,True,True,True,True
223,PPCG0004a,PPCG0004,Primary,0.478,1.689,False,PPCG,.,True,True,True,True
224,PPCG0005a,PPCG0005,Primary,0.673,2.000,False,PPCG,SV,True,True,True,False


***Checking donors / samples have mutations***


In [28]:
muts = load_mutations()
muts = preprocess_mutations(muts)
donors1 = set(hframe[hframe['handling']!='ignore']['donor'].unique())
donors2 = set(muts['donor'].unique())
print(len(donors1))
print(len(donors2))
print(donors1-donors2)
print(donors2-donors1)
print()

samples1 = set(sheet[sheet[['SNV', 'INDEL', 'CNA', 'SV']].any(axis=1)]['sample'].unique())
samples2 = set(muts['sample'].unique())
print(len(samples1))
print(len(samples2))
print(samples1-samples2)
print(samples2-samples1)
sheet[sheet['sample'].isin(samples1-samples2)][['sample', 'tissue', 'cohort', 'Missing_Files', 'SNV', 'INDEL', 'CNA', 'SV']]

982
981
{'PPCG0213'}
set()

1153
1149
{'PPCG0056h', 'PPCG0846c', 'PPCG0058h', 'PPCG0213a'}
set()


,sample,tissue,cohort,Missing_Files,SNV,INDEL,CNA,SV
283,PPCG0056h,Primary,PPCG,"INDEL,SNV,SV",False,False,True,False
295,PPCG0058h,Primary,PPCG,"INDEL,SNV,SV",False,False,True,False
445,PPCG0213a,Primary,PPCG,"INDEL,SV",True,False,True,False
978,PPCG0846c,Primary,PPCG,"INDEL,SV",True,False,True,False


***Assignment method implementations***

fields:
- ***variant_ccf***: the estimated value from the variant call. 
- ***variant_origin***: primary or secondary
- ***variant_ctype***: Trunk, Branch, Seed or Leaf
- ***dpclust_ccf***: the DPClust CCF (in that sample) lifted after assigning the mutation to a clone.
- ***dpclust_origin***: primary or secondary
- ***dpclust_ctype***: the DPClust 'Cluster_Type' (Trunk, Branch, Seed or Leaf)
- ***dpclust_clone***: the DPClust clone
- ***seeding_trajectory***: whether the mutation happened before a seeding event.

1. variant_lineage
2. dpclust_clone
3. dpclust_ccf
4. dpclust_ctype
5. dpclust_lineage

rename fields


----
# Variant fields
----

In [29]:
def assign_single_primary(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    df = table.copy()
    df['variant_origin'] = 'Primary'
    mask = df['variant_ccf']>=0.9
    df.loc[mask, 'variant_ctype'] = 'Trunk'
    df.loc[~mask, 'variant_ctype'] = 'Leaf'
    return df

# Test run
muts = load_mutations()
muts = preprocess_mutations(muts)
muts = muts[muts['donor']=='PPCG0001'].copy()
df = assign_single_primary(muts, hframe)
print(df.drop_duplicates('ident').groupby('variant_origin')['variant_ctype'].value_counts())
df.head()

variant_origin  variant_ctype
Primary         Leaf             34
                Trunk            11
Name: count, dtype: int64


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype
52168,PPCG0001a,PPCG0001,11:5153383,GCC,G,INDEL,INDEL,frameshift_variant,0.880,OR52A5,PPCG,Primary,PPCG0001|INDEL|11:5153383,PPCG0001|INDEL,Primary,Leaf
9449,PPCG0001a,PPCG0001,4:186071244,GCC,G,INDEL,INDEL,3_prime_UTR_variant,1.000,SLC25A4,PPCG,Primary,PPCG0001|INDEL|4:186071244,PPCG0001|INDEL,Primary,Trunk
52235,PPCG0001a,PPCG0001,10:75562095,C,T,SNV,SNV,3_prime_UTR_variant,0.880,NDST2,PPCG,Primary,PPCG0001|SNV|10:75562095,PPCG0001|SNV,Primary,Leaf
30793,PPCG0001a,PPCG0001,11:76174922,C,T,SNV,SNV,missense_variant,1.000,EMSY,PPCG,Primary,PPCG0001|SNV|11:76174922,PPCG0001|SNV,Primary,Trunk
50413,PPCG0001a,PPCG0001,12:109251359,T,A,SNV,SNV,5_prime_UTR_variant,0.890,SSH1,PPCG,Primary,PPCG0001|SNV|12:109251359,PPCG0001|SNV,Primary,Leaf


In [30]:
def assign_single_met(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    df = table.copy()
    mask = df['variant_ccf']>=0.9
    df.loc[mask, 'variant_origin'] = 'Primary'
    df.loc[~mask, 'variant_origin'] = 'Metastasis'
    df.loc[mask, 'variant_ctype'] = 'Trunk'
    df.loc[~mask, 'variant_ctype'] = 'Leaf'
    return df

# Test run
muts = load_mutations()
muts = preprocess_mutations(muts)
muts = muts[muts['donor']=='PPCG0105'].copy()
df = assign_single_met(muts, hframe)
print(df.drop_duplicates('ident').groupby('variant_origin')['variant_ctype'].value_counts())
df.head()

variant_origin  variant_ctype
Metastasis      Leaf             68
Primary         Trunk            58
Name: count, dtype: int64


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype
169587,PPCG0105a,PPCG0105,13:50202435-50208008,.,.,CNA,CNA↓↓,CNAgene,1.000,ARL11,COMBI,Metastasis,PPCG0105|CNA|ARL11,PPCG0105|CNA,Primary,Trunk
172060,PPCG0105a,PPCG0105,10:89511269-89601100,.,.,CNA,CNA↓↓,CNAgene,1.000,ATAD1,COMBI,Metastasis,PPCG0105|CNA|ATAD1,PPCG0105|CNA,Primary,Trunk
169615,PPCG0105a,PPCG0105,13:49882786-50018200,.,.,CNA,CNA↓↓,CNAgene,1.000,CAB39L,COMBI,Metastasis,PPCG0105|CNA|CAB39L,PPCG0105|CNA,Primary,Trunk
169621,PPCG0105a,PPCG0105,13:49822061-49867621,.,.,CNA,CNA↓↓,CNAgene,1.000,CDADC1,COMBI,Metastasis,PPCG0105|CNA|CDADC1,PPCG0105|CNA,Primary,Trunk
169642,PPCG0105a,PPCG0105,13:49227847-49285362,.,.,CNA,CNA↓↓,CNAgene,1.000,CYSLTR2,COMBI,Metastasis,PPCG0105|CNA|CYSLTR2,PPCG0105|CNA,Primary,Trunk


In [31]:
# TODO clean this its scuffed. 
def assign_multi_primary(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    
    def make_judgement(row: pd.Series) -> str:
        if row['n_samples_obs'] == row['n_samples_total']:
            return 'Trunk'
        elif row['n_samples_obs'] >= 2:
            return 'Branch'
        elif row['n_samples_obs'] == 1:
            return 'Leaf'
        raise NotImplementedError(row)
    
    df = table.copy()
    df['variant_origin'] = 'Primary'
    
    sframe = pd.DataFrame(index=sorted(df['ident'].unique())).reset_index()
    sframe = sframe.rename(columns={'index': 'ident'})
    i2l_LUT = df.drop_duplicates('ident').set_index('ident')['label'].to_dict()
    i2o_LUT = df.groupby('ident')['sample'].nunique().to_dict()
    l2e_LUT = hframe.set_index('label')['samples'].to_dict()

    sframe['label'] = sframe['ident'].map(i2l_LUT)
    sframe['n_samples_obs'] = sframe['ident'].map(i2o_LUT)
    sframe['n_samples_total'] = sframe['label'].map(l2e_LUT)
    sframe['verdict'] = sframe.apply(make_judgement, axis=1)
    assert sframe.isna().sum().sum() == 0 
    
    mapper = sframe.set_index('ident')['verdict'].to_dict()
    df['variant_ctype'] = df['ident'].map(mapper)
    return df 

# Test run
muts = load_mutations()
muts = preprocess_mutations(muts)
muts = muts[muts['donor']=='PPCG0056'].copy()
df = assign_multi_primary(muts, hframe)
print(df.drop_duplicates('ident').groupby('variant_origin')['variant_ctype'].value_counts())
df.head()

variant_origin  variant_ctype
Primary         Branch           153
                Leaf              72
                Trunk             44
Name: count, dtype: int64


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype
187961,PPCG0056g,PPCG0056,18:56148479-56296323,.,.,CNA,CNA↓↓,CNAgene,0.110,ALPK2,PPCG,Primary,PPCG0056|CNA|ALPK2,PPCG0056|CNA,Primary,Branch
181189,PPCG0056c,PPCG0056,18:56148479-56296323,.,.,CNA,CNA↓↓,CNAgene,0.220,ALPK2,PPCG,Primary,PPCG0056|CNA|ALPK2,PPCG0056|CNA,Primary,Branch
183837,PPCG0056a,PPCG0056,18:56148479-56296323,.,.,CNA,CNA↓↓,CNAgene,0.200,ALPK2,PPCG,Primary,PPCG0056|CNA|ALPK2,PPCG0056|CNA,Primary,Branch
183875,PPCG0056a,PPCG0056,18:43906772-44043103,.,.,CNA,CNA↓↓,CNAgene,0.200,ARK2C,PPCG,Primary,PPCG0056|CNA|ARK2C,PPCG0056|CNA,Primary,Branch
185869,PPCG0056c,PPCG0056,18:43906772-44043103,.,.,CNA,CNA↓↓,CNAgene,0.170,ARK2C,PPCG,Primary,PPCG0056|CNA|ARK2C,PPCG0056|CNA,Primary,Branch


In [32]:
def assign_multi_met(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    df = table.copy()
    
    i2s_obs_LUT = df.groupby('ident')['sample'].nunique()
    l2s_tot_LUT = hframe.set_index('label')['secondary'].to_dict()
    
    ident2origin = dict()
    ident2ctype = dict()

    for ident in df['ident'].unique():
        label = '|'.join(ident.split('|')[:2])
        n_samples_obs = i2s_obs_LUT[ident]
        n_samples_tot = l2s_tot_LUT[label]
        if n_samples_obs == n_samples_tot:
            ident2ctype[ident] = 'Trunk'
            ident2origin[ident] = 'Primary'
        elif n_samples_obs >= 2:
            ident2ctype[ident] = 'Branch'
            ident2origin[ident] = 'Metastasis'
        elif n_samples_obs == 1:
            ident2ctype[ident] = 'Leaf'
            ident2origin[ident] = 'Metastasis'
        else:
            raise NotImplementedError

    df['variant_origin'] = df['ident'].map(ident2origin)
    df['variant_ctype'] = df['ident'].map(ident2ctype)
    return df

# Test run
muts = load_mutations()
muts = preprocess_mutations(muts)
muts = muts[muts['donor']=='PPCG0102'].copy()
df = assign_multi_met(muts, hframe)
print(df.drop_duplicates('ident').groupby('variant_origin')['variant_ctype'].value_counts())
df.head()

variant_origin  variant_ctype
Metastasis      Leaf             299
                Branch            89
Primary         Trunk            592
Name: count, dtype: int64


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype
160707,PPCG0102a,PPCG0102,8:26605667-26724790,.,.,CNA,CNA↓↓,CNAgene,1.000,ADRA1A,COMBI,Metastasis,PPCG0102|CNA|ADRA1A,PPCG0102|CNA,Primary,Trunk
160692,PPCG0102f,PPCG0102,8:26605667-26724790,.,.,CNA,CNA↓↓,CNAgene,1.000,ADRA1A,COMBI,Metastasis,PPCG0102|CNA|ADRA1A,PPCG0102|CNA,Primary,Trunk
160691,PPCG0102c,PPCG0102,8:26605667-26724790,.,.,CNA,CNA↓↓,CNAgene,1.000,ADRA1A,COMBI,Metastasis,PPCG0102|CNA|ADRA1A,PPCG0102|CNA,Primary,Trunk
160714,PPCG0102e,PPCG0102,8:26605667-26724790,.,.,CNA,CNA↓↓,CNAgene,1.000,ADRA1A,COMBI,Metastasis,PPCG0102|CNA|ADRA1A,PPCG0102|CNA,Primary,Trunk
160709,PPCG0102d,PPCG0102,8:26605667-26724790,.,.,CNA,CNA↓↓,CNAgene,1.000,ADRA1A,COMBI,Metastasis,PPCG0102|CNA|ADRA1A,PPCG0102|CNA,Primary,Trunk


In [33]:
def assign_single_primary_single_met(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    df = table.copy()

    primary_ids = set(df[df['tissue']=='Primary']['ident'].unique())
    secondary_ids = set(df[df['tissue']=='Metastasis']['ident'].unique())

    ident2origin = dict()
    ident2ctype = dict()

    for ident in df['ident'].unique():
        if ident in primary_ids and ident in secondary_ids:
            ident2ctype[ident] = 'Trunk'
            ident2origin[ident] = 'Primary'
        elif ident in primary_ids:
            ident2ctype[ident] = 'Leaf'
            ident2origin[ident] = 'Primary'
        elif ident in secondary_ids:
            ident2ctype[ident] = 'Leaf'
            ident2origin[ident] = 'Metastasis'
        else:
            raise NotImplementedError

    df['variant_origin'] = df['ident'].map(ident2origin)
    df['variant_ctype'] = df['ident'].map(ident2ctype)
    return df

# Test run
muts = load_mutations()
muts = preprocess_mutations(muts)
muts = muts[muts['donor']=='PPCG1058'].copy()
df = assign_single_primary_single_met(muts, hframe)
print(df.drop_duplicates('ident').groupby('variant_origin')['variant_ctype'].value_counts())
df.head()

variant_origin  variant_ctype
Metastasis      Leaf              6
Primary         Leaf             80
                Trunk            15
Name: count, dtype: int64


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype
162596,PPCG1058a,PPCG1058,7:156432975-156469824,.,.,CNA,CNA↓↓,CNAgene,1.000,RNF32,COMBI,Primary,PPCG1058|CNA|RNF32,PPCG1058|CNA,Primary,Leaf
18903,PPCG1058a,PPCG1058,1:158262154,AG,A,INDEL,INDEL,stop_gained,1.000,CD1C,COMBI,Primary,PPCG1058|INDEL|1:158262154,PPCG1058|INDEL,Primary,Leaf
18310,PPCG1058a,PPCG1058,1:19200968,G,GC,INDEL,INDEL,frameshift_variant,1.000,ALDH4A1,COMBI,Primary,PPCG1058|INDEL|1:19200968,PPCG1058|INDEL,Primary,Leaf
37243,PPCG1058a,PPCG1058,3:108677532,AT,A,INDEL,INDEL,3_prime_UTR_variant,0.970,MORC1,COMBI,Primary,PPCG1058|INDEL|3:108677532,PPCG1058|INDEL,Primary,Leaf
8421,PPCG1058a,PPCG1058,5:133292672,G,GTCCT,INDEL,INDEL,frameshift_variant,1.000,C5orf15,COMBI,Primary,PPCG1058|INDEL|5:133292672,PPCG1058|INDEL,Primary,Leaf


In [34]:
def assign_single_primary_multi_met(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    df = table.copy()

    primary_ids = set(df[df['tissue']=='Primary']['ident'].unique())
    l2s_tot_LUT = hframe.set_index('label')['secondary'].to_dict()
    i2s_obs_LUT = df[df['tissue']=='Metastasis'].groupby('ident')['sample'].nunique()
    
    ident2origin = dict()
    ident2ctype = dict()

    for ident in df['ident'].unique():
        label = '|'.join(ident.split('|')[:2])
        n_secondaries_obs = i2s_obs_LUT.get(ident, 0)
        n_secondaries_tot = l2s_tot_LUT[label]
        assert n_secondaries_tot >= 2
        
        if ident in primary_ids and n_secondaries_obs==n_secondaries_tot:
            ident2ctype[ident] = 'Trunk'
            ident2origin[ident] = 'Primary'
        elif ident in primary_ids and n_secondaries_obs>=1:
            ident2ctype[ident] = 'Branch'
            ident2origin[ident] = 'Primary'
        elif ident in primary_ids:
            ident2ctype[ident] = 'Leaf'
            ident2origin[ident] = 'Primary'
        elif n_secondaries_obs==n_secondaries_tot:
            ident2ctype[ident] = 'Seed'
            ident2origin[ident] = 'Primary'
        elif n_secondaries_obs>=2:
            ident2ctype[ident] = 'Branch'
            ident2origin[ident] = 'Metastasis'
        elif n_secondaries_obs==1:
            ident2ctype[ident] = 'Leaf'
            ident2origin[ident] = 'Metastasis'
        else:
            raise NotImplementedError

    df['variant_origin'] = df['ident'].map(ident2origin)
    df['variant_ctype'] = df['ident'].map(ident2ctype)
    return df

# Test run
muts = load_mutations()
muts = preprocess_mutations(muts)
# muts = muts[muts['donor']=='PPCG0388'].copy()
muts = muts[muts['donor']=='PPCG0086'].copy()
df = assign_single_primary_multi_met(muts, hframe)
print(df.drop_duplicates('ident').groupby('variant_origin')['variant_ctype'].value_counts())
df.head()

variant_origin  variant_ctype
Metastasis      Leaf             238
                Branch           151
Primary         Trunk            287
                Leaf             125
                Branch            73
                Seed              10
Name: count, dtype: int64


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype
172573,PPCG0086d,PPCG0086,10:22610028-22620413,.,.,CNA,CNA↓↓,CNAgene,1.000,BMI1,COMBI,Metastasis,PPCG0086|CNA|BMI1,PPCG0086|CNA,Metastasis,Leaf
172574,PPCG0086d,PPCG0086,10:22605315-22609235,.,.,CNA,CNA↓↓,CNAgene,1.000,COMMD3,COMBI,Metastasis,PPCG0086|CNA|COMMD3,PPCG0086|CNA,Metastasis,Leaf
180838,PPCG0086d,PPCG0086,8:33448848-33457541,.,.,CNA,CNA↓↓,CNAgene,0.230,DUSP26,COMBI,Metastasis,PPCG0086|CNA|DUSP26,PPCG0086|CNA,Metastasis,Leaf
180842,PPCG0086d,PPCG0086,8:33228342-33330940,.,.,CNA,CNA↓↓,CNAgene,0.230,FUT10,COMBI,Metastasis,PPCG0086|CNA|FUT10,PPCG0086|CNA,Metastasis,Leaf
180840,PPCG0086d,PPCG0086,8:33342700-33358780,.,.,CNA,CNA↓↓,CNAgene,0.230,MAK16,COMBI,Metastasis,PPCG0086|CNA|MAK16,PPCG0086|CNA,Metastasis,Leaf


***Perform Annotation***

In [35]:
# fmapper = {
#     'single_primary': assign_single_primary,
#     'multi_primary': assign_multi_primary,
#     'single_primary_single_met': assign_single_primary_single_met,
#     'single_primary_multi_met': assign_single_primary_multi_met,
#     'single_met': assign_single_met,
#     'multi_met': assign_multi_met,
# }

# muts = load_mutations()
# muts = preprocess_mutations(muts)
# # muts = muts[muts['donor']=='PPCG0388'].copy()
# # muts = muts[muts['donor']=='PPCG0427'].copy()

# table = pd.DataFrame(columns=muts.columns.to_list() + ['variant_origin', 'variant_ctype'])
# for handling, func in fmapper.items():
#     cigs = set(hframe[hframe['handling']==handling]['label'].unique())
#     df = muts[muts['label'].isin(cigs)].copy()
#     df = func(df, hframe)
#     table = pd.concat([table, df], ignore_index=True)

# assert muts['ident'].nunique() == table['ident'].nunique()
# assert table[table['vclass']=='SV']['variant_ccf'].isna().all()
# assert table[table['vclass']!='SV']['variant_ccf'].notna().all()
# assert table['variant_origin'].isna().sum()==0
# assert table['variant_ctype'].isna().sum()==0
# table = table.rename(columns={'ident': 'variant'})
# table.head()

----
# DPClust fields
----

In [36]:
sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
DONOR2PRIMARY = sheet[sheet['tissue']=='Primary'].groupby('donor')['sample'].agg(set).to_dict()

def load_dpclust_ccfs(filepath: str) -> tuple[pd.DataFrame, dict[str, str]]:
    df = pd.read_csv(filepath, sep=',', header=0)
    df['Cluster'] = df['Cluster'].apply(lambda x: str(x).replace('_', '').split('.')[0])
    df = df.set_index('Cluster')
    tmapper = df['Cluster_Type'].to_dict()
    df.columns = [x.replace('_DNA', '') for x in df.columns]
    df = df[[x for x in df.columns if x.startswith('PPCG')]].copy()
    return df, tmapper

def load_dpclust_asmts(filepath: str) -> dict[str, str]:
    donor = filepath.split('/')[-1][:8]
    df = pd.read_csv(filepath, sep=',', header=0)
    df = df.dropna(subset=['Cluster'])
    df['Cluster'] = df['Cluster'].apply(lambda x: str(x).replace('_', '').split('.')[0])
    df['donor'] = donor
    df['chr'] = df['chr'].astype(str)
    df['pos'] = df['pos'].astype(str)
    df['coords'] = df['chr'] + ':' + df['pos']
    df['ident'] = df['donor'] + '|SNV|' + df['coords']
    return df.set_index('ident')['Cluster'].to_dict()

def assign_dpclust(table: pd.DataFrame, hframe: pd.DataFrame) -> pd.DataFrame:
    # this are filepaths for DPClust / CONIPHER
    donor2ccfpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")}
    donor2asmtpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_SNV_CCF_Cluster_assignment.csv")}
    # donor2treepath = {x.split('/')[-2][:8]: x for x in glob(f"{INDIR_CONIPHER}/*/allTrees.txt")}

    # these are the donors to annotate 
    mut_donors = set(table['donor'].unique())
    dpc_donors = set(donor2ccfpath.keys())
    todo_donors = mut_donors & dpc_donors

    comp = table[~table['donor'].isin(todo_donors)].copy()
    todo = table[table['donor'].isin(todo_donors)].copy()

    # non-DPClust donors just get pd.NA values
    for field in ['dpclust_ccf','dpclust_origin','dpclust_ctype','dpclust_clone']:
        comp[field] = pd.NA

    # iterate through donors to reduce recalculation. 
    for i, donor in enumerate(sorted(todo_donors)):
        # print(i, donor)
        df = todo[todo['donor']==donor].copy()
        asmts = load_dpclust_asmts(donor2asmtpath[donor])
        ccfs, tmapper = load_dpclust_ccfs(donor2ccfpath[donor])
        df = _process_donor(df, donor, asmts, ccfs, tmapper, hframe)
        comp = pd.concat([comp, df], ignore_index=True)

    return comp

def _process_donor(df: pd.DataFrame, donor: str, asmts: dict[str, str], ccfs: pd.DataFrame, tmapper: dict[str, str], hframe: pd.DataFrame) -> pd.DataFrame:
    var2clone = _assign_clones(df, asmts, ccfs, hframe)
    clone2origin = _get_clone2origin_map(ccfs, donor, tmapper)
    clone2ctype = tmapper 
    dpclust_samples = ccfs.columns.to_list()

    for field in ['dpclust_ccf','dpclust_origin','dpclust_ctype','dpclust_clone']:
        df[field] = pd.NA
    
    for idx, row in df.iterrows():
        sample = row['sample']
        clone = var2clone[row['ident']]
        if clone is None or sample not in dpclust_samples:
            continue 

        ccf = float(ccfs.loc[clone, row['sample']])
        df.loc[idx, 'dpclust_clone'] = clone
        df.loc[idx, 'dpclust_origin'] = clone2origin[clone]
        df.loc[idx, 'dpclust_ctype'] = clone2ctype[clone]
        df.loc[idx, 'dpclust_ccf'] = ccf

    return df.copy()

def _get_clone2origin_map(ccfs: pd.DataFrame, donor: str, tmapper: dict[str, str]) -> dict[str, str]:
    IGNORE_THRESH = 0.025
    primaries = list(DONOR2PRIMARY.get(donor, {}))
    primary = primaries[0] if len(primaries)==1 else None
    primary = primary if primary in ccfs.columns else None

    if primary is None:
        priclones = set()
    else:
        priclones = set((ccfs[ccfs[primary]>=IGNORE_THRESH]).index.to_list())

    # print(f'\n=== primary: {primary} ===')
    # print(f'=== priclones: {priclones} ===\n')

    mapper = {}
    for clone in ccfs.index.to_list():
        if clone in tmapper and tmapper[clone] in ['Trunk', 'Seed']:
            mapper[clone] = 'Primary'
        elif clone in priclones:
            mapper[clone] = 'Primary'
        else:
            mapper[clone] = 'Metastasis'
    return mapper 

def _assign_clones(muts: pd.DataFrame, asmts: dict[str, str], ccfs: pd.DataFrame, hframe: pd.DataFrame) -> dict[str, str]:
    var2clone = {}

    # check clones
    asmt_clones = sorted(set(asmts.values())) 
    ccf_clones = sorted(ccfs.index.to_list())
    if asmt_clones != ccf_clones:
        print('\nasmt clones != ccf clones')
        print(f"asmt: {asmt_clones}")
        print(f"ccf:  {ccf_clones}\n")
    
    # perform direct assignment of SNVs clustered by DPClust
    complete = set()
    for var, clone in asmts.items():
        if clone not in ccf_clones:
            continue 
        complete.add(var)
        var2clone[var] = clone
    
    # remaining SNV/INDEL/SV/CNA variants use the method below. 
    # in short, variant ccfs (across all samples) are compared to 
    # DPClust cluster ccfs and the best match is chosen. 
    df = muts[~muts['ident'].isin(complete)].copy()
    df = df.set_index('sample')
    mask = df['vclass']=='SV'
    df.loc[mask, 'variant_ccf'] = 1.00
    assert df['variant_ccf'].isna().sum() == 0 

    # this is a lookup table which maps donor:vclass to the list of samples for which we had data. 
    label2samples = hframe.set_index('label')['sample_list'].to_dict()
    # we group variants into chunks by vclass (PPCG0086:SNV, PPCG0086:INDEL, ... etc)
    # this is because some donors are missing files for a particular sample.  
    # eg PPCG0086a might have had SNV,INDEL,CNA data but no SV.vcf file. 
    # this means we should remove the PPCG0086a sample from the DPClust ccf dataframe. 
    # otherwise, if a cluster is present in PPCG0086a, it gets an unfair penalty for all SV
    # variant calculations because it looks like PPCG0086a doesn't have the SV but actually
    # it may indeed have, we just don't have a record of it because PPCG0086a had no file. 
    label_chunks = {l: dfslice for l, dfslice in df.groupby('label')}
    for label, lslice in label_chunks.items():
        local_samples = label2samples[label]
        local_ccfs = ccfs.T.copy()
        if local_ccfs.shape[0] > len(local_samples):
            print(f"{label}: removing {set(local_ccfs.index)-set(local_samples)} from consideration.")
            local_ccfs = local_ccfs.loc[local_samples].copy()

        # we then process each variant individually as a dataframe chunk. 
        # each record relates to that variant in different samples. 
        var_chunks = {v: dfslice for v, dfslice in lslice.groupby('ident')}
        for var, vslice in var_chunks.items():
            # dis da magique funkshion
            clone = _process_variant_chunk(vslice, local_ccfs)
            # if clone is None:
            #     continue 
            # update maps.
            var2clone[var] = clone

    return var2clone

def _process_variant_chunk(df: pd.DataFrame, local_ccfs: pd.DataFrame) -> str|None:
    # early exit edge case. 
    # this can happen if a variant is observed in a sample(s) which is not present 
    # in DPClust cluster ccfs. Ie DPClust was run without the sample. 
    # just return None and handle on the other end. 
    # var_samples = set(df.index)
    # dpc_samples = set(local_ccfs.index)
    for sample in df.index:
        if sample not in local_ccfs.index:
            # df.drop(sample)
            return None
    
    # if here, we can calculate the fitness of each clone. 
    vseries = (local_ccfs.loc[df.index.to_list()]>0.0).all()
    # strict_clones = vseries[vseries==True].index.to_list()

    ccfs = local_ccfs.copy()
    ccfs['observed'] = df['variant_ccf'].to_dict()
    ccfs['observed'] = ccfs['observed'].fillna(0.0)

    # sum absolute difference between expected and observed ccf across all samples
    # extra penalty when variant is observed in a sample but DPClust says the clone does not exist in that sample. 
    scores = []
    for clone in ccfs.columns:
        if clone == 'observed':  # don't compare to self OFC.
            continue

        diff = (ccfs[clone]-ccfs['observed']).abs().sum()
        penalty = ((ccfs[clone]==0.0) & (ccfs['observed']>0.0)).sum() * 10.0
        score = diff + penalty
        scores.append((clone, diff, penalty, score))
    
    # pick best clone
    scores = sorted(scores, key=lambda x: x[-1])
    clone = scores[0][0]
    
    # if clone != '21':
    #     print()
    #     print(ccfs)
    #     print()
    #     for item in scores:
    #         print(f"clone={item[0]}, diff={round(item[1], 2)}, penalty={round(item[2], 2)}, score={round(item[3], 2)}")
    
    return clone 


***Perform annotation and DPClust assignment***

In [37]:
fmapper = {
    'single_primary': assign_single_primary,
    'multi_primary': assign_multi_primary,
    'single_primary_single_met': assign_single_primary_single_met,
    'single_primary_multi_met': assign_single_primary_multi_met,
    'single_met': assign_single_met,
    'multi_met': assign_multi_met,
}

muts = load_mutations()
muts = preprocess_mutations(muts)
# muts = muts[muts['donor']=='PPCG0395'].copy()
# muts = muts[muts['donor'].isin(['PPCG0086', 'PPCG0001'])].copy()

table = pd.DataFrame(columns=muts.columns.to_list() + ['variant_origin', 'variant_ctype'])
for handling, func in fmapper.items():
    cigs = set(hframe[hframe['handling']==handling]['label'].unique())
    df = muts[muts['label'].isin(cigs)].copy()
    if df.shape[0] == 0:
        continue
    df = func(df, hframe)
    table = pd.concat([table, df], ignore_index=True)

table = assign_dpclust(table, hframe)
assert muts['ident'].nunique() == table['ident'].nunique()
assert table[table['vclass']=='SV']['variant_ccf'].isna().all()
assert table[table['vclass']!='SV']['variant_ccf'].notna().all()
assert table['variant_origin'].isna().sum()==0
assert table['variant_ctype'].isna().sum()==0
# table = table.rename(columns={'ident': 'variant'})
table[table['cohort']=='COMBI'].head()

PPCG0179|INDEL: removing {'PPCG0179j'} from consideration.
PPCG0395|SV: removing {'PPCG0395c'} from consideration.
PPCG0425|SV: removing {'PPCG0425a'} from consideration.
PPCG0428|INDEL: removing {'PPCG0428d', 'PPCG0428c'} from consideration.
PPCG0428|SV: removing {'PPCG0428d', 'PPCG0428c'} from consideration.

asmt clones != ccf clones
asmt: ['11', '21', '31', '4']
ccf:  ['21', '31', '4']

PPCG0442|SV: removing {'PPCG0442c'} from consideration.


,sample,donor,coords,ref,alt,vclass,vtype,annotation,variant_ccf,gene,cohort,tissue,ident,label,variant_origin,variant_ctype,dpclust_ccf,dpclust_origin,dpclust_ctype,dpclust_clone
182184,PPCG1058a,PPCG1058,7:156432975-156469824,.,.,CNA,CNA↓↓,CNAgene,1.000,RNF32,COMBI,Primary,PPCG1058|CNA|RNF32,PPCG1058|CNA,Primary,Leaf,<NA>,<NA>,<NA>,<NA>
182185,PPCG1058a,PPCG1058,1:158262154,AG,A,INDEL,INDEL,stop_gained,1.000,CD1C,COMBI,Primary,PPCG1058|INDEL|1:158262154,PPCG1058|INDEL,Primary,Leaf,<NA>,<NA>,<NA>,<NA>
182186,PPCG1058a,PPCG1058,1:19200968,G,GC,INDEL,INDEL,frameshift_variant,1.000,ALDH4A1,COMBI,Primary,PPCG1058|INDEL|1:19200968,PPCG1058|INDEL,Primary,Leaf,<NA>,<NA>,<NA>,<NA>
182187,PPCG1058a,PPCG1058,3:108677532,AT,A,INDEL,INDEL,3_prime_UTR_variant,0.970,MORC1,COMBI,Primary,PPCG1058|INDEL|3:108677532,PPCG1058|INDEL,Primary,Leaf,<NA>,<NA>,<NA>,<NA>
182188,PPCG1058a,PPCG1058,5:133292672,G,GTCCT,INDEL,INDEL,frameshift_variant,1.000,C5orf15,COMBI,Primary,PPCG1058|INDEL|5:133292672,PPCG1058|INDEL,Primary,Leaf,<NA>,<NA>,<NA>,<NA>


***Final Formatting***

In [63]:
muts = load_mutations()

# derive identifier 'ident' for each variant
mask = muts['vclass'].isin(['SV', 'CNA'])
muts.loc[mask, 'ident'] = muts['donor'] + '|' + muts['vclass'] + '|' + muts['gene']
muts.loc[~mask, 'ident'] = muts['donor'] + '|' + muts['vclass'] + '|' + muts['coords']
muts = muts.drop_duplicates(subset=['sample', 'vclass', 'coords', 'gene'])

# lift variant annotations
master = muts.copy()
master = master.rename(columns={'est_ccf': 'variant_ccf'})
table_cpy = table.copy()
for field in [
    'variant_origin', 'variant_ctype', 'dpclust_clone', 
    'dpclust_ccf', 'dpclust_origin', 'dpclust_ctype'
]:
    mapper = table_cpy.drop_duplicates('ident').set_index('ident')[field].to_dict()
    master[field] = master['ident'].map(mapper)

print(muts['ident'].nunique())
print(master['ident'].nunique())
print(table_cpy['ident'].nunique())

# add new pretty ids
master = master.sort_values(['cohort', 'sample', 'coords'])
idmap = dict()
for donor in master['donor'].unique():
    temp = master[master['donor']==donor].drop_duplicates('ident')
    i = 0
    for ident in temp['ident'].to_list():
        i += 1
        if i < 10:
            new_id = f"{donor}:000{i}"
        elif i < 100:
            new_id = f"{donor}:00{i}"
        elif i < 1000:
            new_id = f"{donor}:0{i}"
        elif i < 10000:
            new_id = f"{donor}:{i}"
        else:
            raise ValueError
        idmap[ident] = new_id 
master['ID'] = master['ident'].map(idmap)

# final field order
fields = [
    'ID', 'cohort', 'sample', 'tissue', 
    'gene', 'coords', 'ref', 'alt', 'vclass', 'vtype', 'annotation', 
    'variant_ccf', 'variant_origin', 'variant_ctype', 
    'dpclust_clone', 'dpclust_ccf', 'dpclust_origin', 'dpclust_ctype', 
]
master = master[fields].copy()
master['annotation'] = master['annotation'].replace('CNAgene', 'deep_deletion')
master = master.rename(columns={
    'variant_ccf': 'var_ccf', 
    'variant_origin': 'var_origin', 
    'variant_ctype': 'var_ctype', 
    'dpclust_clone': 'DPC_clone', 
    'dpclust_ccf': 'DPC_ccf', 
    'dpclust_origin': 'DPC_origin', 
    'dpclust_ctype': 'DPC_ctype', 
})
master = master.sort_values('ID')
master.to_csv(OUTFILE_MASTER, sep='\t', index=False, float_format='%.2f')
master.head()


217729
217729
217729


,ID,cohort,sample,tissue,gene,coords,ref,alt,vclass,vtype,annotation,var_ccf,var_origin,var_ctype,DPC_clone,DPC_ccf,DPC_origin,DPC_ctype
256666,PPCG0001:0001,PPCG,PPCG0001a,Primary,NDST2,10:75562095,C,T,SNV,SNV,3_prime_UTR_variant,0.880,Primary,Leaf,NaN,NaN,NaN,NaN
56687,PPCG0001:0002,PPCG,PPCG0001a,Primary,ADK,10:76221633-10:115874382,.,.,SV,INV,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
265522,PPCG0001:0003,PPCG,PPCG0001a,Primary,OR52A5,11:5153383,GCC,G,INDEL,INDEL,frameshift_variant,0.880,Primary,Leaf,NaN,NaN,NaN,NaN
56676,PPCG0001:0004,PPCG,PPCG0001a,Primary,DGCR8,11:65251003-22:20095850,.,.,SV,TRA,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
256667,PPCG0001:0005,PPCG,PPCG0001a,Primary,EMSY,11:76174922,C,T,SNV,SNV,missense_variant,1.000,Primary,Trunk,NaN,NaN,NaN,NaN


In [60]:
temp = master[master['cohort']=='PPCG']
temp[temp.duplicated('ID', keep=False)].head(10)

# print(temp.drop_duplicates('ID').groupby('donor')['vclass'].value_counts().unstack().fillna(0).astype(int))



,ID,cohort,donor,sample,tissue,gene,coords,ref,alt,vclass,vtype,annotation,var_ccf,var_origin,var_ctype,DPC_clone,DPC_ccf,DPC_origin,DPC_ctype
56686,PPCG0001:0009,PPCG,PPCG0001,PPCG0001a,Primary,COL4A5,X:99022512-X:107890320,.,.,SV,INV,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56685,PPCG0001:0009,PPCG,PPCG0001,PPCG0001a,Primary,COL4A5,14:38056130-X:107911449,.,.,SV,TRA,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56672,PPCG0001:0009,PPCG,PPCG0001,PPCG0001a,Primary,COL4A5,X:99022533-X:107883918,.,.,SV,INV,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56684,PPCG0001:0009,PPCG,PPCG0001,PPCG0001a,Primary,COL4A5,14:38056099-X:107884021,.,.,SV,TRA,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56679,PPCG0001:0022,PPCG,PPCG0001,PPCG0001a,Primary,ERG,21:39872654-21:42874716,.,.,SV,DEL,gene_fusion&frameshift_variant,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56682,PPCG0001:0022,PPCG,PPCG0001,PPCG0001a,Primary,ERG,21:39872643-21:42838202,.,.,SV,INV,bidirectional_gene_fusion,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56680,PPCG0001:0023,PPCG,PPCG0001,PPCG0001a,Primary,TMPRSS2,21:39872654-21:42874716,.,.,SV,DEL,gene_fusion&frameshift_variant,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56681,PPCG0001:0023,PPCG,PPCG0001,PPCG0001a,Primary,TMPRSS2,21:42838178-21:42874828,.,.,SV,INV,bidirectional_gene_fusion,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56683,PPCG0001:0023,PPCG,PPCG0001,PPCG0001a,Primary,TMPRSS2,21:39872643-21:42838202,.,.,SV,INV,bidirectional_gene_fusion,NaN,Primary,Leaf,NaN,NaN,NaN,NaN
56688,PPCG0001:0035,PPCG,PPCG0001,PPCG0001a,Primary,PTPRK,6:128350767-6:131439588,.,.,SV,INV,transcript_ablation,NaN,Primary,Leaf,NaN,NaN,NaN,NaN


In [40]:
keepfields = [
    'ID', 'cohort', 'donor', 'sample', 'tissue', 
    'gene', 'coords', 'ref', 'alt', 'vclass', 'vtype', 'annotation', 
    'variant_ccf', 'variant_origin', 'variant_ctype', 
    'dpclust_clone', 'dpclust_ccf', 'dpclust_origin', 'dpclust_ctype', 
]
master = table.copy()
master = master.sort_values(['cohort', 'sample', 'coords'])
idmap = dict()
for donor in master['donor'].unique():
    temp = master[master['donor']==donor].drop_duplicates('ident')
    i = 0
    for ident in temp['ident'].to_list():
        i += 1
        if i < 10:
            new_id = f"{donor}:000{i}"
        elif i < 100:
            new_id = f"{donor}:00{i}"
        elif i < 1000:
            new_id = f"{donor}:0{i}"
        elif i < 10000:
            new_id = f"{donor}:{i}"
        else:
            raise ValueError
        idmap[ident] = new_id 
master['ID'] = master['ident'].map(idmap)
master = master[keepfields].copy()
master['annotation'] = master['annotation'].replace('CNAgene', 'deep_deletion')
master = master.rename(columns={
    'variant_ccf': 'var_ccf', 
    'variant_origin': 'var_origin', 
    'variant_ctype': 'var_ctype', 
    'dpclust_clone': 'DPC_clone', 
    'dpclust_ccf': 'DPC_ccf', 
    'dpclust_origin': 'DPC_origin', 
    'dpclust_ctype': 'DPC_ctype', 
})
master.head()


,ID,cohort,donor,sample,tissue,gene,coords,ref,alt,vclass,vtype,annotation,var_ccf,var_origin,var_ctype,DPC_clone,DPC_ccf,DPC_origin,DPC_ctype
187779,PPCG0086:0001,COMBI,PPCG0086,PPCG0086a,Metastasis,CNNM1,10:101106103-10:105176747,.,.,SV,DUP,gene_fusion&frameshift_variant,NaN,Primary,Seed,1,0.990,Primary,Trunk
188344,PPCG0086:0002,COMBI,PPCG0086,PPCG0086a,Metastasis,PDCD11,10:101106103-10:105176747,.,.,SV,DUP,gene_fusion&frameshift_variant,NaN,Primary,Seed,1,0.990,Primary,Trunk
188730,PPCG0086:0003,COMBI,PPCG0086,PPCG0086a,Metastasis,VTI1A,10:114255864-10:114448539,.,.,SV,DEL,gene_fusion&frameshift_variant,NaN,Primary,Trunk,1,0.990,Primary,Trunk
187723,PPCG0086:0004,COMBI,PPCG0086,PPCG0086a,Metastasis,CCDC186,10:115910077-10:119069253,.,.,SV,DUP,gene_fusion,NaN,Primary,Trunk,1,0.990,Primary,Trunk
188355,PPCG0086:0005,COMBI,PPCG0086,PPCG0086a,Metastasis,PDZD8,10:115910077-10:119069253,.,.,SV,DUP,gene_fusion,NaN,Primary,Trunk,1,0.990,Primary,Trunk


***Sanity checks & Debugging***

In [38]:
dpc_donors = set([x.split('/')[-1][:8] for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")])
temp = table[table['donor'].isin(dpc_donors)]
temp = temp.drop_duplicates('ident')
temp = temp[temp['dpclust_origin'].notna()]

vcounts = temp['donor'].value_counts().to_dict()
mask_o = temp['variant_origin']==temp['dpclust_origin']
mask_c = temp['variant_ctype']==temp['dpclust_ctype']
mask_b = mask_o & mask_c
sframe = pd.DataFrame(index=sorted(dpc_donors))
sframe['# variants'] = vcounts
sframe['agree origin'] = temp[mask_o]['donor'].value_counts()
sframe['agree ctype'] = temp[mask_c]['donor'].value_counts()
sframe['agree both'] = temp[mask_b]['donor'].value_counts()
sframe = sframe.fillna(0).astype(int)
sframe['origin (%)'] = sframe['agree origin'] / sframe['# variants'] * 100
sframe['ctype (%)'] = sframe['agree ctype'] / sframe['# variants'] * 100
sframe['agree both (%)'] = sframe['agree both'] / sframe['# variants'] * 100
sframe['disagree both (%)'] = 100 - (sframe['agree both'] / sframe['# variants'] * 100)
sframe = sframe.fillna(0).astype(int)
sframe = sframe.sort_values('agree both (%)')
sframe.head(20)


,# variants,agree origin,agree ctype,agree both,origin (%),ctype (%),agree both (%),disagree both (%)
PPCG1062,129,128,12,12,99,9,9,90
PPCG0428,216,203,41,41,93,18,18,81
PPCG0388,183,134,72,72,73,39,39,60
PPCG1082,293,280,128,128,95,43,43,56
PPCG0434,184,148,107,107,80,58,58,41
PPCG0395,104,102,63,63,98,60,60,39
PPCG1026,109,94,67,67,86,61,61,38
PPCG0424,222,177,146,146,79,65,65,34
PPCG1078,148,124,119,100,83,80,67,32
PPCG0988,319,319,218,218,100,68,68,31


In [65]:
temp = table[table['donor']=='PPCG0395']
temp = temp.sort_values('ident')
print(temp.drop_duplicates('ident').groupby('dpclust_origin')['variant_origin'].value_counts(dropna=False))
print()
print(temp.drop_duplicates('ident').groupby('dpclust_ctype')['variant_ctype'].value_counts(dropna=False))
print()
print(temp.drop_duplicates('ident').groupby('dpclust_origin')['dpclust_clone'].value_counts(dropna=False))
print()
print(temp.groupby('ident')['sample'].agg(set).value_counts())
print()
print(temp['ident'].value_counts().value_counts())
print()
print(temp[temp['dpclust_ctype']!=temp['variant_ctype']]['dpclust_clone'].value_counts())
temp[(temp['vclass']=='SNV') & (temp['dpclust_ctype']!=temp['variant_ctype'])].head(10)

dpclust_origin  variant_origin
Metastasis      Metastasis         7
Primary         Primary           95
                Metastasis         2
Name: count, dtype: int64

dpclust_ctype  variant_ctype
Leaf           Leaf             63
Seed           Leaf             41
Name: count, dtype: int64

dpclust_origin  dpclust_clone
Metastasis      3                 7
Primary         1                56
                2                41
Name: count, dtype: int64

sample
{PPCG0395a}    95
{PPCG0395c}     9
Name: count, dtype: int64

count
1    104
Name: count, dtype: int64

dpclust_clone
2    41
Name: count, dtype: int64


,ident,gene,vclass,label,cohort,donor,sample,tissue,variant_ccf,variant_origin,variant_ctype,dpclust_ccf,dpclust_origin,dpclust_ctype,dpclust_clone
22,PPCG0395|SNV|11:116693391,APOA4,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,1.000,Primary,Leaf,1.000,Primary,Seed,2
31,PPCG0395|SNV|13:113540863,ATP11A,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395c,Metastasis,0.660,Metastasis,Leaf,0.300,Primary,Seed,2
32,PPCG0395|SNV|13:31233874,USPL1,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.630,Primary,Leaf,1.000,Primary,Seed,2
33,PPCG0395|SNV|14:101204199,DLK1,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.760,Primary,Leaf,1.000,Primary,Seed,2
41,PPCG0395|SNV|17:7163700,CLDN7,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.820,Primary,Leaf,1.000,Primary,Seed,2
43,PPCG0395|SNV|18:10704403,PIEZO2,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,1.000,Primary,Leaf,1.000,Primary,Seed,2
44,PPCG0395|SNV|18:11609640,SLC35G4,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.890,Primary,Leaf,1.000,Primary,Seed,2
46,PPCG0395|SNV|19:52115165,SIGLEC5,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.850,Primary,Leaf,1.000,Primary,Seed,2
58,PPCG0395|SNV|22:26908469,TFIP11,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.920,Primary,Leaf,1.000,Primary,Seed,2
59,PPCG0395|SNV|22:30752870,SF3A1,SNV,PPCG0395|SNV,COMBI,PPCG0395,PPCG0395a,Primary,0.820,Primary,Leaf,1.000,Primary,Seed,2


In [67]:
# this are filepaths for DPClust / CONIPHER
donor2ccfpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")}
donor2asmtpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_SNV_CCF_Cluster_assignment.csv")}
donor2treepath = {x.split('/')[-2][:8]: x for x in glob(f"{INDIR_CONIPHER}/*/allTrees.txt")}

dpc_donors = set([x.split('/')[-1][:8] for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")])

for donor in sorted(list(dpc_donors)):
    # asmts = load_dpclust_asmts_hack(donor2asmtpath[donor])
    ccfs = load_dpclust_ccfs_hack(donor2ccfpath[donor])
    ctypes = set(ccfs['Cluster_Type'].unique())
    if 'Trunk' not in ctypes:
        print(donor)
    # tdict = load_conipher_trees(donor2treepath[donor])
    # # tdict = label_conipher_trees1(tdict, ccfs)
    # print(ccfs)
    # draw_conipher_trees_alt(tdict, ccfs)

PPCG0395
PPCG0425
PPCG0427
PPCG0434
PPCG0442
PPCG1072


In [ ]:
pfields = ['gene', 'vclass', 'sample', 'tissue', 'variant_ccf', 'dpclust_ccf', 'variant_origin', 'dpclust_origin', 'variant_ctype', 'dpclust_ctype', 'dpclust_clone']
temp = table[table['donor']=='PPCG0086']
temp[(temp['dpclust_ctype']=='Trunk') & (temp['variant_ctype']!='Trunk')][pfields]

In [22]:
import numpy as np 
import networkx as nx 
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout

def draw_ccfs(ccfs: pd.DataFrame) -> None:
    df = ccfs.drop('Cluster_Type', axis=1)
    print()
    for clone, row in df.iterrows():
        print(clone, '\t', row[row>0].index.to_list())

def draw_conipher_trees_alt(trees: dict[str, nx.DiGraph], ccfs: pd.DataFrame) -> None:
    T = trees['tree 1']
    df = ccfs.drop('Cluster_Type', axis=1)
    samples = df.columns.to_list()
    
    i = 0
    ncols = 4 if len(samples) >= 4 else len(samples) % 4
    nrows = len(samples) // 4 
    if len(samples) % 4 != 0:
        nrows += 1
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 4*nrows))
    for sample in samples:
        if nrows > 1:
            row = i // 4
            col = i % 4 
            ax = axes[row, col]
        else:
            col = i % 4 
            ax = axes[col]

        pos = graphviz_layout(T, prog='dot')
        present = df[df[sample]>0].index.to_list()
        nl_present = [n for n in T.nodes() if n in present]
        nl_missing = [n for n in T.nodes() if n not in present]
        nx.draw_networkx_nodes(T, pos, nodelist=nl_present, node_color='skyblue', node_size=500, ax=ax)
        nx.draw_networkx_nodes(T, pos, nodelist=nl_missing, node_color='salmon', node_size=500, ax=ax)
        nx.draw_networkx_labels(T, pos, ax=ax)
        nx.draw_networkx_edges(T, pos, ax=ax)
        ax.set_title(sample)
        i += 1
    plt.show()
    plt.close()


sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
sheet = sheet.sort_values('sample')

# this are filepaths for DPClust / CONIPHER
donor2ccfpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")}
donor2asmtpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_SNV_CCF_Cluster_assignment.csv")}
donor2treepath = {x.split('/')[-2][:8]: x for x in glob(f"{INDIR_CONIPHER}/*/allTrees.txt")}

tree_donors = set(donor2treepath.keys())
dpc_donors = set(donor2ccfpath.keys())
target_donors = tree_donors & dpc_donors
donor2primaries = sheet[sheet['tissue']=='Primary'].groupby('donor')['sample'].agg(set).to_dict()

target_donors = ['PPCG0086']
for donor in sorted(target_donors): 
    asmts = load_dpclust_asmts_hack(donor2asmtpath[donor])
    ccfs = load_dpclust_ccfs_hack(donor2ccfpath[donor])
    tdict = load_conipher_trees(donor2treepath[donor])
    tdict = label_conipher_trees1(tdict, ccfs)
    draw_conipher_trees_alt(tdict, ccfs)


# logic based (variant_ccf, variant_origin, variant_ctype): 
# - strictly about which samples the variant is observed in. 
# - eg COMBI donors with 2 mets; variant observed in all samples:       'Primary Trunk'
# - eg COMBI donors with 2 mets; variant observed in primary & one met: 'Primary Seed'
# - eg COMBI donors with 2 mets; variant observed in one met:           'Metastasis Leaf'
# - 'variant_ccf' only used for...
#   - PPCG donors with single primary: >=0.9 deemed 'Primary Trunk', <0.9 'Primary Leaf'
#   - COMBI donors with single met:    >=0.9 deemed 'Primary Trunk', <0.9 'Metastasis Leaf'

# clone assignment
# strict: only consider clones 
# 

   

NameError: name 'load_dpclust_asmts_hack' is not defined

In [ ]:
print(table.drop_duplicates(subset=['dpclust_ccf', 'dpclust_origin', 'dpclust_ctype', 'dpclust_clone']))
print()
print(table.groupby('dpclust_origin')['dpclust_clone'].agg(set))
print()
print(table.groupby('dpclust_origin')['dpclust_ctype'].agg(set))
print()
print(table.groupby(['dpclust_clone', 'sample'])['dpclust_ccf'].agg(set))

***Perform Annotation***

In [ ]:
# def _process_variant_chunk(df: pd.DataFrame, local_ccfs: pd.DataFrame) -> str|None:
#     # early exit edge case. 
#     # this can happen if a variant is observed in a sample(s) which is not present 
#     # in DPClust cluster ccfs. Ie DPClust was run without the sample. 
#     # just return None and handle on the other end. 
#     df_samples = set(df.index)
#     ccf_samples = set(local_ccfs.index)
#     if len(df_samples-ccf_samples) != 0:
#         return None

#     vseries = (local_ccfs.loc[list(df_samples)]>0.0).all()
#     valid = vseries[vseries==True].index.to_list()
#     if len(valid) == 0:
#         print(df)
#         print()
#         print(local_ccfs)
#         print()
#         raise NotImplementedError
#     elif len(valid) == 1:
#         return valid[0]

#     scores = []
#     ccfs = local_ccfs.copy()
#     ccfs['observed'] = df['variant_ccf'].to_dict()
#     ccfs['observed'] = ccfs['observed'].fillna(0.0)
    
#     for clone in valid:
#         if clone == 'observed':  # don't compare to self OFC.
#             continue 
#         # sum absolute difference between expected and observed ccf across all samples
#         # extra penalty for situations involving ccf=0.0  
#         # - clone not in sample, but variant is detected (-1 score because this feels bad)
#         # - clone in sample, but variant not detected (-0.5 score because somewhat explainable)
#         diff = (ccfs[clone]-ccfs['observed']).abs().sum()
#         penalty1 = ((ccfs[clone]==0.0) & (ccfs['observed']>0.0)).sum() * 1.0
#         penalty2 = ((ccfs[clone]>0.0) & (ccfs['observed']==0.0)).sum() * 0.5
#         score = diff + penalty1 + penalty2
#         scores.append((clone, float(round(score, 2))))
#         # print(f"clone={clone}, diff={round(diff, 2)}, penalty1={penalty1}, penalty2={penalty2}, score={round(score, 2)}")
    
#     # pick best clone
#     scores = sorted(scores, key=lambda x: x[1])
#     clone = scores[0][0]
#     return clone 


In [ ]:
# dpclust_ccf
# dpclust_origin
# dpclust_ctype
# dpclust_clone



In [ ]:
print(table.groupby('handling')['donor'].nunique())
print()
print(table.drop_duplicates('variant').groupby(['handling', 'donor', 'vclass'])['lineage'].value_counts().unstack().fillna(0).astype(int).head(10))

***Validation & Sanity Checks***

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx 
from networkx.drawing.nx_agraph import graphviz_layout

def draw_conipher_trees(trees: dict[str, nx.DiGraph]) -> None:
    if len(trees) == 1:
        name, T = list(trees.items())[0]
        fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
        pos = graphviz_layout(T, prog='dot')
        nl_trunk = [n for n in T.nodes() if T.nodes[n]['label']=='Trunk']
        nl_branch = [n for n in T.nodes() if T.nodes[n]['label']=='Branch']
        nl_leaf = [n for n in T.nodes() if T.nodes[n]['label']=='Leaf']
        nl_seed = [n for n in T.nodes() if T.nodes[n]['label']=='Seed']
        nl_seeding = [n for n in T.nodes() if T.nodes[n]['label']=='Seeding']
        nl_unknown = [n for n in T.nodes() if T.nodes[n]['label']=='Unknown']
        nx.draw_networkx_nodes(T, pos, nodelist=nl_trunk, node_color='salmon', node_size=500, ax=ax)
        nx.draw_networkx_nodes(T, pos, nodelist=nl_branch, node_color='khaki', node_size=500, ax=ax)
        nx.draw_networkx_nodes(T, pos, nodelist=nl_seed, node_color='skyblue', node_size=500, ax=ax)
        nx.draw_networkx_nodes(T, pos, nodelist=nl_leaf, node_color='palegreen', node_size=500, ax=ax)
        nx.draw_networkx_nodes(T, pos, nodelist=nl_seeding, node_color='red', node_size=500, ax=ax)
        nx.draw_networkx_nodes(T, pos, nodelist=nl_unknown, node_color='khaki', node_size=500, ax=ax)
        nx.draw_networkx_labels(T, pos, ax=ax)
        nx.draw_networkx_edges(T, pos, ax=ax)
        ax.set_title(name)
    else:
        i = 0
        ncols = 4 if len(trees) >= 4 else len(trees) % 4
        nrows = len(trees) // 4 
        if len(trees) % 4 != 0:
            nrows += 1
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 4*nrows))
        for name, T in trees.items():
            if nrows > 1:
                row = i // 4
                col = i % 4 
                ax = axes[row, col]
            else:
                col = i % 4 
                ax = axes[col]
            pos = graphviz_layout(T, prog='dot')
            nl_trunk = [n for n in T.nodes() if T.nodes[n]['label']=='Trunk']
            nl_branch = [n for n in T.nodes() if T.nodes[n]['label']=='Branch']
            nl_leaf = [n for n in T.nodes() if T.nodes[n]['label']=='Leaf']
            nl_seed = [n for n in T.nodes() if T.nodes[n]['label']=='Seed']
            nl_seeding = [n for n in T.nodes() if T.nodes[n]['label']=='Seeding']
            nl_unknown = [n for n in T.nodes() if T.nodes[n]['label']=='Unknown']
            nx.draw_networkx_nodes(T, pos, nodelist=nl_trunk, node_color='salmon', node_size=500, ax=ax)
            nx.draw_networkx_nodes(T, pos, nodelist=nl_branch, node_color='khaki', node_size=500, ax=ax)
            nx.draw_networkx_nodes(T, pos, nodelist=nl_seed, node_color='skyblue', node_size=500, ax=ax)
            nx.draw_networkx_nodes(T, pos, nodelist=nl_leaf, node_color='palegreen', node_size=500, ax=ax)
            nx.draw_networkx_nodes(T, pos, nodelist=nl_seeding, node_color='red', node_size=500, ax=ax)
            nx.draw_networkx_nodes(T, pos, nodelist=nl_unknown, node_color='khaki', node_size=500, ax=ax)
            nx.draw_networkx_labels(T, pos, ax=ax)
            nx.draw_networkx_edges(T, pos, ax=ax)
            ax.set_title(name)
            i += 1
    plt.show()
    plt.close()


In [23]:
def load_dpclust_asmts_hack(filepath: str):
    donor = filepath.split('/')[-1][:8]
    df = pd.read_csv(filepath, sep=',', header=0)
    df = df.dropna(subset=['Cluster'])
    df['Cluster'] = df['Cluster'].apply(lambda x: str(x).replace('_', '').split('.')[0])
    df['donor'] = donor
    df['chr'] = df['chr'].astype(str)
    df['pos'] = df['pos'].astype(str)
    df['coords'] = df['chr'] + ':' + df['pos']
    df['ident'] = df['donor'] + '|SNV|' + df['coords']
    return df

def load_dpclust_ccfs_hack(filepath: str):
    df = pd.read_csv(filepath, sep=',', header=0)
    df['Cluster'] = df['Cluster'].apply(lambda x: str(x).replace('_', '').split('.')[0])
    df = df.set_index('Cluster')
    df.columns = [x.replace('_DNA', '') for x in df.columns]
    df = df[[x for x in df.columns if x.startswith('PPCG') or x=='Cluster_Type']].copy()
    return df

def load_conipher_trees(filepath: str) -> dict[str, nx.DiGraph]:
    trees = {}
    
    with open(filepath, 'r') as fp:
        line = fp.readline().strip()
        line = fp.readline().strip()
        assert line == '# tree 1'

        name = line.strip('# ')
        T = nx.DiGraph()
        line = fp.readline().strip()
        while line:
            if line.startswith('#'):
                trees[name] = T
                name = line.strip('# ')
                T = nx.DiGraph()
            else:
                parent, child = line.split('\t')
                parent = parent.replace('_', '')
                child = child.replace('_', '')
                T.add_edge(parent, child)
            line = fp.readline().strip()
        trees[name] = T
    
    return trees 


In [ ]:
def label_conipher_trees1(trees: dict[str, nx.DiGraph], ccfs: pd.DataFrame) -> dict[str, nx.DiGraph]:
    # approach 1: directly lift dpclust ccf 'Cluster_Type' annotations.
    tmapper = ccfs['Cluster_Type'].to_dict()
    for name, T in trees.items():
        for node in T.nodes():
            T.nodes[node]['label'] = tmapper.get(node, 'Unknown')
    return trees 

# def label_conipher_trees2(trees: dict[str, nx.DiGraph], ccfs: pd.DataFrame, primary_sample_id: str|None) -> dict[str, nx.DiGraph]:
#     for name, T in trees.items():
#         labeller = TreeLabeller(T, ccfs, primary_sample_id)
#         labeller.label()
#     return trees 

def label_conipher_trees2(trees: dict[str, nx.DiGraph], ccfs: pd.DataFrame, primary_sample_id: str|None) -> dict[str, nx.DiGraph]:
    for name, T in trees.items():
        labeller = SeedingLabeller(T, ccfs, primary_sample_id)
        labeller.label()
    return trees

class SeedingLabeller:
    IGNORE_TRESH = 0.025
    CLONAL_TRESH = 0.900

    def __init__(self, T: nx.DiGraph, ccfs: pd.DataFrame, primary_sample_id: str|None):
        self.T = T 
        self.ccfs = ccfs
        self.primary_sample_id = primary_sample_id
    
    def label(self) -> nx.DiGraph:
        if self.primary_sample_id is None:
            seeding = self.seeding_noprimary
        else:
            seeding = self.seeding_normal

        T = self.T
        for node in T.nodes():
            if node in seeding:
                T.nodes[node]['label'] = 'Seeding'
            else:
                T.nodes[node]['label'] = 'Unknown'
        return T

    @property
    def primary_clones(self) -> set[str]:
        if self.primary_sample_id is None:
            return set()
        return set((self.ccfs[self.ccfs[self.primary_sample_id]>=self.IGNORE_TRESH]).index.to_list())
    
    @property
    def trunk(self) -> str:
        return list(nx.topological_sort(self.T))[0]

    @property
    def leaves(self) -> set[str]:
        ccfs_t = self.ccfs.drop('Cluster_Type', axis=1).T
        T = self.T
        
        leaves = set()
        for node in T.nodes():
            if T.out_degree(node) == 0:
                # edge case: PPCG0435 clone 11 missing from DPClust CCFs.
                # don't do the secondary check. 
                if node not in self.ccfs.index:
                    leaves.add(node)
                    continue

                samples = (ccfs_t[ccfs_t[node]>=self.IGNORE_TRESH]).index.to_list()
                if len(samples) == 1:
                    leaves.add(node)
        return leaves

    @property
    def seeding_noprimary(self) -> set[str]:
        tmapper = self.ccfs['Cluster_Type'].to_dict()
        ccfs_t = self.ccfs.drop(['Cluster_Type'], axis=1).T
        trunk = self.trunk
        leaves = self.leaves
        
        out = set()
        for clone in ccfs_t.columns:
            if clone==trunk or tmapper[clone] in ['Trunk', 'Seed']:
                out.add(clone)
                continue
            # if clone in leaves or tmapper[clone]=='Leaf':
            #     continue
            
            # n_clonal = len(ccfs_t[ccfs_t[clone]>=self.CLONAL_TRESH].index.to_list())
            # n_detected = len(ccfs_t[ccfs_t[clone]>=self.IGNORE_TRESH].index.to_list())
            # is_seeding = True if n_clonal==n_detected or tmapper[clone]=='Trunk' else False
            # print(f"cluster {clone}: n_clonal={n_clonal}, n_detected={n_detected}, seeding={is_seeding}")
            # if is_seeding:
            #     out.add(clone)
        return out 
    
    @property
    def seeding_normal(self) -> set[str]:
        # leaves are not seeding (private primary/secondary)
        # trunk is seeding 
        # internal nodes are seeding if 
        # 1: clonal in 1+ secondary samples and detected in primary, or 
        # 2: clonal in 2+ secondary samples and parent is detected in the primary

        T = self.T
        trunk = self.trunk
        leaves = self.leaves
        priclones = self.primary_clones
        secccfs = self.ccfs.drop([self.primary_sample_id, 'Cluster_Type'], axis=1).T
        tmapper = self.ccfs['Cluster_Type'].to_dict()
        
        out = set()
        for clone in self.ccfs.index:
            if clone==trunk or tmapper[clone] in ['Trunk', 'Seed']:
                out.add(clone)
                continue
            if clone in leaves or tmapper[clone]=='Leaf':
                continue 
            if clone not in T.nodes():
                raise NotImplementedError(clone)
            
            secondaries = set(secccfs[secccfs[clone]>=self.CLONAL_TRESH].index.to_list())
            if len(secondaries) >= 1 and clone in priclones:
                out.add(clone)
            if len(secondaries) >= 2:
                parents = list(T.predecessors(clone))
                if len(parents)==1 and parents[0] in priclones:
                    out.add(clone)

        return out



In [ ]:
sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
sheet = sheet.sort_values('sample')

# this are filepaths for DPClust / CONIPHER
donor2ccfpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_Cluster_CCFs.csv")}
donor2asmtpath = {x.split('/')[-1][:8]: x for x in glob(f"{INDIR_DPCLUST}/*_SNV_CCF_Cluster_assignment.csv")}
donor2treepath = {x.split('/')[-2][:8]: x for x in glob(f"{INDIR_CONIPHER}/*/allTrees.txt")}

tree_donors = set(donor2treepath.keys())
dpc_donors = set(donor2ccfpath.keys())
target_donors = tree_donors & dpc_donors
donor2primaries = sheet[sheet['tissue']=='Primary'].groupby('donor')['sample'].agg(set).to_dict()

target_donors = ['PPCG0086']
for donor in sorted(target_donors): 
    asmts = load_dpclust_asmts_hack(donor2asmtpath[donor])
    ccfs = load_dpclust_ccfs_hack(donor2ccfpath[donor])

    df_samples = set(['PPCG0086c', 'PPCG0086e'])    
    local_ccfs = ccfs.drop('Cluster_Type', axis=1).T
    print(local_ccfs)
    print()
    print(local_ccfs.loc[list(df_samples)]>0.0)
    print()
    vseries = (local_ccfs.loc[list(df_samples)]>0.0).all()
    valid = vseries[vseries==True].index.to_list()
    print(valid)
    # valid_clones = local_ccfs.loc[list(df_samples)]
    break 

    primaries = list(donor2primaries.get(donor, {}))
    primary = primaries[0] if len(primaries)==1 else None 
    primary = primary if primary in ccfs.columns else None
    print(donor)
    print(ccfs)
    tdict = load_conipher_trees(donor2treepath[donor])
    tdict = label_conipher_trees1(tdict, ccfs)
    draw_conipher_trees(tdict)
    tdict = label_conipher_trees2(tdict, ccfs, primary)
    draw_conipher_trees(tdict)


In [ ]:
donors = [
    'PPCG0088', # primary missing
    'PPCG0089', # primary missing
    'PPCG0102', # primary missing
    'PPCG0182', # primary missing
    'PPCG0183', # primary missing
    'PPCG0388', # primary missing from DPClust
    'PPCG0390', # primary missing
    'PPCG0425', # primary missing
]
for donor in donors:
    sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
    sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
    sheet = sheet.sort_values('sample')
    print()
    print(donor)
    print(sheet[sheet['donor']==donor])


In [ ]:

# class TreeLabeller:
#     CCF_THRESH = 0.025
#     CLONAL_TRESH = 0.900

#     def __init__(self, T: nx.DiGraph, ccfs: pd.DataFrame, primary_sample_id: str|None):
#         self.T = T 
#         self.ccfs = ccfs
#         self.primary_sample_id = primary_sample_id
    
#     def label(self) -> nx.DiGraph:
#         # print()
#         # print(f"primary: {self.primary_sample_id}")
#         # print(f"primary_clones: {sorted(self.primary_clones)}")
#         # print(f"secondary_clones: {sorted(self.secondary_clones)}")
#         # print()
#         # print(f"trunk: {self.trunk}")
#         # print(f"leaves: {sorted(self.leaves)}")
#         # print(f"seeds: {sorted(self.seeds)}")
#         # print()
#         # print(self.ccfs)
#         trunk = self.trunk
#         leaves = self.leaves
#         # seeds = self.seeds
#         seeding = self.seeding
#         # mettraj = self.primary_clones & self.secondary_clones

#         T = self.T
#         for node in T.nodes():
#             if node in seeding:
#                 T.nodes[node]['label'] = 'Seeding'
#             # elif node == trunk:
#             #     T.nodes[node]['label'] = 'Trunk'
#             elif node in leaves:
#                 T.nodes[node]['label'] = 'Leaf'
#             # elif node in seeds:
#             #     T.nodes[node]['label'] = 'Seed'
#             else:
#                 T.nodes[node]['label'] = 'Unknown'
#         return T

#     @property
#     def primary_clones(self) -> set[str]:
#         if self.primary_sample_id is None:
#             return set()
#         return set((self.ccfs[self.ccfs[self.primary_sample_id]>self.CCF_THRESH]).index.to_list())
    
#     @property
#     def secondary_clones(self) -> set[str]:
#         clones = set()
#         # ccfs dataframe but only for secondary samples
#         ccfs = self.ccfs.drop([self.primary_sample_id, 'Cluster_Type'], axis=1)
#         # transpose so clusters are columns, then count number of samples with CCF>0
#         ccfs_t = ccfs.T 
#         for clust in ccfs_t.columns:
#             samples = (ccfs_t[ccfs_t[clust]>self.CCF_THRESH]).index.to_list()
#             if len(samples) > 0:
#                 clones.add(clust)
#         return clones 
    
#     @property
#     def trunk(self) -> str:
#         return list(nx.topological_sort(self.T))[0]

#     @property
#     def leaves(self) -> set[str]:
#         ccfs_t = self.ccfs.drop('Cluster_Type', axis=1).T
#         T = self.T
        
#         leaves = set()
#         for node in T.nodes():
#             if T.out_degree(node) == 0:
#                 # edge case: PPCG0435 clone 11 missing from DPClust CCFs.
#                 # don't do the secondary check. 
#                 if node not in self.ccfs.index:
#                     leaves.add(node)
#                     continue

#                 samples = (ccfs_t[ccfs_t[node]>self.CCF_THRESH]).index.to_list()
#                 if len(samples) == 1:
#                     leaves.add(node)
#         return leaves

    
#     @property
#     def seeds(self) -> set[str]:
#         # traceback from each secondary Leaf until hit a clone in the primary. 
#         # if the identified clone is the Trunk, usually this traceback would be ignored. 
#         # however, there is an 'undetected' edge case (eg PPCG0086 clone 15) to address.
#         # clone 15 has CCF=0.0 in the primary sample, but probably was just missed. 
#         # either due to being at very low CCF, or only being present in an unsampled region of the prostate.
#         # logic: if the traceback destination is the Trunk clone, yet the penultimate clone in the traceback 
#         # is present in 2+ met samples, the penultimate clone is the seed. Looks like only present in mets, but actually
#         # probably just missed in primary. 
        
#         T = self.T
#         trunk = self.trunk
#         leaves = self.leaves
#         priclones = self.primary_clones
#         secclones = self.secondary_clones
        
#         # print()
#         # print('--- SEEDS ---')
#         seeds = set()
#         for leaf in leaves:
#             if leaf not in secclones:
#                 # print(f"{leaf}:\t(primary leaf)")
#                 continue 
            
#             node = leaf 
#             path = [node]
#             parents = list(T.predecessors(node))
#             while True:
#                 assert len(parents) <= 1
#                 if len(parents) == 0:
#                     break 
#                 node = parents[0]
#                 if node in priclones:
#                     path.append(node)
#                     break 
#                 path.append(node)
#                 parents = list(T.predecessors(node))

#             # print(f"{leaf}:\t{path}")

#             # normal case: terminal node in path is not the Trunk
#             query = path[-1]
#             if query != trunk:
#                 seeds.add(query)
            
#             # undetected edge case: terminal node is Trunk
#             query = path[-2]
#             if path[-1]==trunk and query not in leaves:
#                 secccfs = self.ccfs.drop([self.primary_sample_id, 'Cluster_Type'], axis=1).T
#                 donor_secondaries = set(secccfs.index.to_list())
#                 clone_secondaries = set(secccfs[secccfs[query]>self.CCF_THRESH].index.to_list())
#                 # print()
#                 # print(secccfs)
#                 # print(donor_secondaries, '(all secondaries)')
#                 # print(clone_secondaries, f'(clone={query} secondaries)')
#                 # if len(clone_secondaries) >= 2 and len(donor_secondaries)!=len(clone_secondaries):
#                 if len(clone_secondaries) >= 2:
#                     seeds.add(query)
            
#         # print()
#         return seeds
    
#     @property
#     def seeding(self) -> set[str]:
#         # trunk is seeding 
#         # leaves are not seeding (private primary/secondary)
#         # internal nodes are seeding if 
#         # 1: clonal in 1+ secondary samples and detected in primary, or 
#         # 2: clonal in 2+ secondary samples and parent is detected in the primary

#         T = self.T
#         trunk = self.trunk
#         leaves = self.leaves
#         priclones = self.primary_clones
#         secclones = self.secondary_clones
#         secccfs = self.ccfs.drop([self.primary_sample_id, 'Cluster_Type'], axis=1).T
#         tmapper = self.ccfs['Cluster_Type'].to_dict()
        
#         out = set()
#         for clone in self.ccfs.index:
#             if clone==trunk or tmapper[clone] in ['Trunk', 'Seed']:
#                 out.add(clone)
#                 continue
#             if clone in leaves or tmapper[clone]=='Leaf':
#                 continue 
#             if clone not in T.nodes():
#                 # if clone 
#                 raise NotImplementedError(clone)
            
#             secondaries = set(secccfs[secccfs[clone]>self.CLONAL_TRESH].index.to_list())
#             if len(secondaries) >= 1 and clone in priclones:
#                 out.add(clone)
#             if len(secondaries) >= 2:
#                 parents = list(T.predecessors(clone))
#                 if len(parents)==1 and parents[0] in priclones:
#                     out.add(clone)

#         return out



In [ ]:
print(table.groupby('cohort')['donor'].nunique())
print()
print(hframe.groupby(['vclass', 'handling'])['donor'].nunique().unstack().fillna(0).astype(int))
print()
print(table.groupby(['vclass', 'handling'])['donor'].nunique().unstack().fillna(0).astype(int))
pfields = ['variant','gene','cohort','tissue','est_ccf','clonality','lineage','tree_clone','handling']
targets = [
    'PPCG0001|INDEL|11:5153383',
    'PPCG0050|SV|ERG',
    'PPCG0056|CNA|ALPK2',
    'PPCG0086|SNV|10:117706080',
    'PPCG0086|SNV|12:70077814'
]
# temp = table[(table['handling']=='dpclust_matching') & (table['vclass']=='SNV')]
# temp[temp.duplicated(subset=['variant'], keep=False)].sort_values('variant')[pfields].head(20)
table[table['variant'].isin(targets)].sort_values('variant')[pfields]

In [ ]:
print(table['gene'].nunique())
print(table['vclass'].value_counts())
counts = table.drop_duplicates('variant').groupby('donor')['vclass'].value_counts().unstack().fillna(0).astype(int)
counts['total'] = counts.sum(axis=1)
counts = counts.sort_values('total', ascending=False)
sns.violinplot(counts)
plt.ylim(-100, 500)
plt.show()
plt.close()
counts.head(10)

In [ ]:
print(table[table['cohort']=='PPCG'].groupby('lineage')['donor'].nunique())
print()
print(table[table['cohort']=='COMBI'].groupby('lineage')['donor'].nunique())

In [ ]:
bins = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
sns.histplot(counts.drop(['total'], axis=1), bins=bins)
# sns.histplot(counts, bins=bins, x='CNA', label='CNA')
# sns.histplot(counts, bins=bins, x='INDEL', label='INDEL')
# sns.histplot(counts, bins=bins, x='SV', label='SV')
# sns.histplot(counts, bins=bins, x='SNV', label='SNV')
plt.legend()
plt.show()
plt.close()
# sns.histplot(table.drop_duplicates('variant'), x='donor')

In [ ]:
counts = table[(table['handling']=='multi_primary') & (table['vclass']=='SV')]
counts.drop_duplicates('variant').groupby('donor')['lineage'].value_counts().unstack().fillna(0).astype(int)

In [ ]:
muts = pd.read_csv(INFILE_MUTATIONS, sep='\t', header=0)
sheet = pd.read_csv(INFILE_SHEET, sep='\t', header=0)
sheet['tissue'] = sheet['tissue'].replace('Recurrence', 'Metastasis')
valid = set(sheet['sample'].unique())
muts = muts[muts['sample'].isin(valid)].copy()

donor2cohort = sheet.drop_duplicates('donor').set_index('donor')['cohort'].to_dict()
sample2tissue = sheet.drop_duplicates('sample').set_index('sample')['tissue'].to_dict()
muts['cohort'] = muts['donor'].map(donor2cohort)
muts['tissue'] = muts['sample'].map(sample2tissue)

muts.head()

In [ ]:
annot = preprocess_mutations(muts)
print(annot.head())

In [ ]:
print(hframe.head())
print()
sheet.head()

In [ ]:
print(table.groupby('handling')['lineage'].value_counts(dropna=False, normalize=True).unstack().fillna(0.0).astype(float))
print()
print(table.groupby('handling')['variant'].nunique())
table.head()

In [ ]:
print(hframe[hframe['label']=='PPCG0050|INDEL'])
print()
print(annot[annot['donor']=='PPCG0050'].drop_duplicates(subset=['donor', 'vclass', 'sample']))
print()
annot[annot['ident']=='PPCG0050|INDEL|KCNH5|14:63170341']

In [ ]:
print(hframe[hframe['label']=='PPCG0050:SNV'])
print()
print(sheet[sheet['donor']=='PPCG0050'])


In [ ]:
temp = df[df['handling'].isin(['COMBI_single_secondary', 'COMBI_multi_secondary'])]

In [ ]:

# print(samples_LUT['PPCG0179:SNV'])
# print(samples_LUT['PPCG0179:INDEL'])
# print(samples_LUT['PPCG0179:CNA'])
# print(samples_LUT['PPCG0179:SV'])

# for vclass in vclasses:
#     for donor
#     samples = samples_LUT[f"{donor}:{vclass}"]
#     tissues = 
#     ppcg_single = sheet[(sheet['donor'].isin(donors_ppcg)) & (sheet['donor'].isin(donors_single))]['donor'].unique()


# counts = sheet.groupby('donor')['sample'].nunique()
# donors_combi = set(sheet[sheet['cohort']=='COMBI']['donor'].unique())
# donors_ppcg = set(sheet[sheet['cohort']=='DONOR']['donor'].unique())
# donors_single = set(counts[counts==1].index.to_list())
# donors_multi = set(counts[counts==1].index.to_list())
# donors_primary = set(sheet[sheet['tissue']=='Primary']['donor'].unique())

# ppcg_single = sheet[(sheet['donor'].isin(donors_ppcg)) & (sheet['donor'].isin(donors_single))]['donor'].unique()
# ppcg_multi = sheet[(sheet['donor'].isin(donors_ppcg)) & (sheet['donor'].isin(donors_multi))]['donor'].unique()
# combi_primary_single = sheet[(sheet['donor'].isin(donors_ppcg)) & (sheet['donor'].isin(donors_single))]['donor'].unique()
# combi_multi = sheet[(sheet['donor'].isin(donors_ppcg)) & (sheet['donor'].isin(donors_multi))]['donor'].unique()


# singles = counts[]
# ppcg_multi = 

In [ ]:
sheet['tissue'].value_counts()

Rules:
- PPCG variants for donors with multiple samples: 
- 

- PPCG variants in multiple samples: clonal if detected in all samples, else subclonal

In [ ]:
counts = annot[(annot['vclass']!='SV') & (annot['cohort']=='PPCG')]['ident'].value_counts()
counts.head()
# counts = annot[annot['cohort']=='PPCG']['ident'].value_counts().head(10)
# counts = counts[counts>=2]

In [ ]:
singles = annot['ident'] 

mask = (annot['cohort']=='PPCG') & (annot[''])
chunks = {ident: records for ident, records in annot.groupby('ident')}
i = 0 
for ident, df in chunks.items():
    i += 1
    if i % 1000 == 0:
        print(f"processed {i}/{len(chunks)} genes...", end='\r')
print(f"processed {i}/{len(chunks)} genes... done.")

In [ ]:

# handle PPCG 
# annot = df[df['cohort']=='PPCG'].copy()
# annot['treeclone'] = pd.NA 
# annot['lineage'] = 'primary'

# handle COMBI

def assign_lineage(df: pd.DataFrame, ident: str, tree_dict: dict) -> pd.DataFrame:
    



# # load all trees
#     donor = ident[:8]
#     if donor in tree_dict:
#         treeclone, lineage = tree_dict[donor]
    
    



# df.head()

# counts = pd.DataFrame(index=sorted(muts['donor'].unique()))
# counts['primary'] = muts[muts['tissue']=='Primary'].groupby('donor')['sample'].nunique()
# counts['secondary'] = muts[muts['tissue'].isin(['Metastasis', 'Recurrence'])].groupby('donor')['sample'].nunique()
# counts = counts.fillna(0).astype(int)
# counts['cohort'] = muts.drop_duplicates('donor').set_index('donor')['cohort'].to_dict()
# print()
# print(counts['cohort'].value_counts(dropna=False))
# print()
# print(counts.tail())



***Older***

In [ ]:
print(muts.shape)
muts = muts.drop_duplicates(subset=['sample', 'vclass', 'coords', 'gene'])
print(muts.shape)

counts = pd.DataFrame(index=sorted(muts['donor'].unique()))
counts['primary'] = muts[muts['tissue']=='Primary'].groupby('donor')['sample'].nunique()
counts['secondary'] = muts[muts['tissue'].isin(['Metastasis', 'Recurrence'])].groupby('donor')['sample'].nunique()
counts = counts.fillna(0).astype(int)
counts['cohort'] = muts.drop_duplicates('donor').set_index('donor')['cohort'].to_dict()
print()
print(counts['cohort'].value_counts(dropna=False))
print()
print(counts.tail())

ppcg_all = set(counts[counts['cohort']=='PPCG'].index.to_list())
combi_pri_met = set(counts[(counts['cohort']=='COMBI') & (counts['primary']>=1) & (counts['secondary']>=1)].index.to_list())
combi_multi_met = set(counts[(counts['cohort']=='COMBI') & (counts['primary']==0) & (counts['secondary']>=2)].index.to_list())
combi_single_met = set(counts[(counts['cohort']=='COMBI') & (counts['primary']==0) & (counts['secondary']==1)].index.to_list())

print()
print(len(ppcg_all))
print(len(combi_pri_met))
print(len(combi_multi_met))
print(len(combi_single_met))
assert len(ppcg_all & combi_pri_met) == 0
assert len(ppcg_all & combi_multi_met) == 0
assert len(ppcg_all & combi_single_met) == 0
assert len(combi_pri_met & combi_multi_met) == 0
assert len(combi_pri_met & combi_single_met) == 0
assert len(combi_multi_met & combi_single_met) == 0

ass = pd.DataFrame(columns=muts.columns.to_list()+['lineage'])
ass.head()


PPCG donors

In [ ]:
def lineage_assignment_

# PPCG donors
df = muts[muts['donor'].isin(ppcg_all)].copy()
df['lineage'] = 'primary'
ass = pd.concat([ass, df], ignore_index=True)
print(ass.groupby('cohort')['donor'].nunique())

COMBI donors with Primary, 1+ Metastasis

In [ ]:
df = muts[muts['donor'].isin(combi_pri_met)].copy()

def assign_lineage(row: pd.Series) -> str:
    if row['primary']==1 and row['secondary']==0:
        return 'primary'
    if row['secondary']==1 and row['primary']==0:
        return 'secondary'
    return 'seeding'

df['ident'] = df['donor'] + '|' + df['coords'] + '|' + df['vclass'] + '|' + df['gene']
sframe = pd.DataFrame(index=sorted(df['ident'].unique()))
sframe['primary'] = df[df['tissue']=='Primary'].groupby('ident')['sample'].nunique()
sframe['secondary'] = df[df['tissue'].isin(['Metastasis', 'Recurrence'])].groupby('ident')['sample'].nunique()
sframe = sframe.fillna(0).astype(int)

sframe['lineage'] = sframe.apply(assign_lineage, axis=1)
print()
print(sframe['lineage'].value_counts())

df['lineage'] = df['ident'].map(sframe['lineage'].to_dict())
df = df.drop('ident', axis=1)
df = df.sort_values(by=['donor', 'vclass', 'coords', 'sample'])

bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(6, 6))
sns.histplot(df[df['lineage']=='primary'], x='est_ccf', bins=bins, ax=axes[0, 0])
sns.histplot(df[df['lineage']=='secondary'], x='est_ccf', bins=bins, ax=axes[0, 1])
sns.histplot(df[(df['lineage']=='seeding') & (df['tissue']=='Primary')], x='est_ccf', bins=bins, ax=axes[1, 0])
sns.histplot(df[(df['lineage']=='seeding') & (df['tissue'].isin(['Metastasis', 'Recurrence']))], x='est_ccf', bins=bins, ax=axes[1, 1])
axes[0, 0].set_title('Primary')
axes[0, 1].set_title('Secondary')
axes[1, 0].set_title('Seeding (prostate samples)')
axes[1, 1].set_title('Seeding (secondary samples)')
plt.tight_layout()
plt.show()
plt.close()

sns.histplot(muts[muts['vclass']=='SV'], x='est_ccf', alpha=0.5, bins=bins, stat='proportion', label='SV')
sns.histplot(muts[muts['vclass']=='SNV'], x='est_ccf', alpha=0.5, bins=bins, stat='proportion', label='SNV')
sns.histplot(muts[muts['vclass']=='CNA'], x='est_ccf', alpha=0.5, bins=bins, stat='proportion', label='CNA')
sns.histplot(muts[muts['vclass']=='INDEL'], x='est_ccf', alpha=0.5, bins=bins, stat='proportion', label='INDEL')
plt.legend()
plt.show()
plt.close()

# df[df['lineage']=='primary'].sort_values('est_ccf').head(10)
ass = pd.concat([ass, df], ignore_index=True)
print(ass.groupby('cohort')['donor'].nunique())


COMBI donors with No Primary, 2+ Metastasis

In [ ]:
# 2+ mets: seeding.
# 1 met and high ccf: seeding
# 1 met and low ccf: secondary 

df = muts[muts['donor'].isin(combi_multi_met)].copy()

df['ident'] = df['donor'] + '|' + df['coords'] + '|' + df['vclass'] + '|' + df['gene']
counts = df.groupby('ident')['sample'].nunique()
multi = (counts[counts>=2].index.to_list())
single = (counts[counts==1].index.to_list())

mask1 = df['ident'].isin(multi)
mask2 = (df['ident'].isin(single)) & (df['est_ccf'].isna())
mask3 = (df['ident'].isin(single)) & (df['est_ccf']>=0.9)
mask4 = (df['ident'].isin(single)) & (df['est_ccf']<0.9)

df = df.drop('ident', axis=1)

df.loc[mask1, 'lineage'] = 'seeding'
df.loc[mask2, 'lineage'] = 'seeding'
df.loc[mask3, 'lineage'] = 'seeding'
df.loc[mask4, 'lineage'] = 'secondary'

ass = pd.concat([ass, df], ignore_index=True)
print(ass.groupby('cohort')['donor'].nunique())


COMBI donors with No Primary, 1 Metastasis

In [ ]:
# if clonal, mark as seeding, else secondary. 
df = muts[muts['donor'].isin(combi_single_met)].copy()
mask1 = df['est_ccf'].isna()
mask2 = df['est_ccf']>=0.9
mask3 = df['est_ccf']<0.9

df.loc[mask1, 'lineage'] = 'seeding'
df.loc[mask2, 'lineage'] = 'seeding'
df.loc[mask3, 'lineage'] = 'secondary'

ass = pd.concat([ass, df], ignore_index=True)
print(ass.groupby('cohort')['donor'].nunique())


In [ ]:
print(muts.shape)
print(ass.shape)
print()
print(muts.groupby('cohort')['donor'].nunique())
print()
print(ass.groupby('cohort')['donor'].nunique())
print()
print(ass[ass['cohort']=='COMBI'].groupby('vclass')['lineage'].value_counts(dropna=False).unstack().fillna(0).astype(int))
ass.head()

In [ ]:
ass['ident'] = ass['cohort'] + ', tissue=' + ass['tissue'] + ', lineage=' + ass['lineage']
ass = ass.sort_values('vclass')
for ident in sorted(ass['ident'].unique()):
    df = ass[(ass['ident']==ident) & (ass['est_ccf'].notna())]
    sns.histplot(df, x='est_ccf', alpha=0.5, bins=10, stat='density', hue='vclass', multiple='dodge', common_norm=False, legend=True)
    plt.title(ident)
    plt.show()
    plt.close()